# Zero-DCE++ SE-Block — MLflow Logging Notebook

Loads your trained model + training history, runs evaluation on the test set,
then logs **everything** to your MLflow server.

**Run order:** top to bottom, all cells in sequence.

**What gets logged to MLflow:**
- Hyperparameters (lr, epochs, batch size, loss weights, model config)
- Per-epoch training loss (total + 4 components) as MLflow step metrics
- Final evaluation: PSNR, SSIM, LPIPS, NIQE
- Training curve plot as artifact
- Model checkpoint as artifact


### 1. Install Dependencies

In [ ]:
!pip install mlflow lpips piq scikit-image -q
print('Done')


### 2. Imports

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import mlflow
import mlflow.pytorch
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

try:
    import lpips as lpips_lib
    lpips_model = lpips_lib.LPIPS(net='alex', verbose=False).to(device)
    lpips_model.eval()
    LPIPS_OK = True
    print('LPIPS loaded')
except Exception:
    LPIPS_OK = False
    print('LPIPS unavailable')

try:
    from piq import niqe
    NIQE_OK = True
    print('NIQE loaded')
except Exception:
    NIQE_OK = False
    print('NIQE unavailable')


### 3. Configuration — Edit paths here

In [ ]:
# ── EDIT THESE ─────────────────────────────────────────────────────────
CHECKPOINT_PATH  = 'best_zerodce_seblock_v2.pth'   # your saved model
HISTORY_PATH     = 'training_history.json'          # see cell below to save history first
TRAINING_PLOT    = 'training_history.png'           # saved by training notebook
LOLV2_PATH       = Path('LOL-v2')                  # dataset root

MLFLOW_URI       = 'http://202.91.9.5:5000'
EXPERIMENT_NAME  = 'Zero-DCE-SE-Block'
RUN_NAME         = 'zerodce_seblock_lolv2_200ep'

# Hyperparameters (must match training)
HPARAMS = {
    'model':           'Zero-DCE++ + SE-Block',
    'dataset':         'LOL-v2 Real_captured',
    'epochs':          200,
    'batch_size':      8,
    'img_size':        512,
    'learning_rate':   1e-4,
    'weight_decay':    1e-4,
    'scheduler':       'CosineAnnealingLR',
    'curve_iterations': 8,
    'w_spa':           1,
    'w_exp':           10,
    'w_col':           5,
    'w_tvA':           1600,
    'exposure_target': 0.6,
    'se_reduction':    8,
    'grad_clip':       0.1,
    'novelty':         'SE-Block (Channel Attention) after Conv3',
}
print('Config ready')


### 4. Save Training History to JSON

Run this cell **once** right after your training notebook finishes (while `history` is still in memory).  
If you already have `training_history.json`, skip this cell.


In [ ]:
# ── Run this in the same kernel as your training notebook ───────────────
# Paste your history dict here if running in a fresh kernel:
# history = {'loss': [...], 'spa': [...], 'exp': [...], 'col': [...], 'tv': [...], 'lr': [...]}

# Otherwise if history is already in memory:
try:
    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f)
    print(f'Saved history to {HISTORY_PATH} ({len(history["loss"])} epochs)')
except NameError:
    print('WARNING: history not in memory.')
    print('Either re-run training first, or manually set history dict above.')


### 5. Model Definition (identical to training)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

class DCENet(nn.Module):
    def __init__(self, use_attention=True):
        super().__init__()
        self.use_attention = use_attention
        self.conv1 = nn.Conv2d(3,  32, 3, 1, 1)
        self.conv2 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv3 = nn.Conv2d(32, 32, 3, 1, 1)
        if use_attention:
            self.se = SEBlock(32, reduction=8)
        self.conv4 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv5 = nn.Conv2d(64, 32, 3, 1, 1)
        self.conv6 = nn.Conv2d(64, 32, 3, 1, 1)
        self.conv7 = nn.Conv2d(64, 24, 3, 1, 1)
        self.relu  = nn.ReLU(inplace=True)
    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(x2))
        if self.use_attention:
            x3 = self.se(x3)
        x4 = self.relu(self.conv4(x3))
        x5 = self.relu(self.conv5(torch.cat([x4, x3], 1)))
        x6 = self.relu(self.conv6(torch.cat([x5, x2], 1)))
        x7 = torch.cat([x6, x1], 1)
        return torch.tanh(self.conv7(x7))

def enhance_image_with_curves(x, alpha):
    enhanced = x.clone()
    for i in range(8):
        a_i = alpha[:, i*3:(i+1)*3, :, :]
        enhanced = enhanced + a_i * enhanced * (1 - enhanced)
    return torch.clamp(enhanced, 0, 1)

# Load checkpoint
model = DCENet(use_attention=True).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params:,} parameters')


### 6. Dataset Loader

In [ ]:
class LOLDataset(Dataset):
    """Full original resolution — no resize for evaluation."""
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        p = self.pairs[idx]
        low    = transforms.ToTensor()(Image.open(p['low']).convert('RGB'))
        normal = transforms.ToTensor()(Image.open(p['normal']).convert('RGB'))
        return {'low': low, 'normal': normal, 'filename': p['filename']}

def load_image_pairs(data_path):
    low_path    = data_path / 'Low'
    normal_path = data_path / 'Normal'
    lows    = sorted([f for f in os.listdir(low_path)    if f.endswith(('.jpg','.png'))])
    normals = sorted([f for f in os.listdir(normal_path) if f.endswith(('.jpg','.png'))])
    assert len(lows) == len(normals), f'Pair mismatch: {len(lows)} vs {len(normals)}'
    return [{'low': str(low_path/l), 'normal': str(normal_path/n), 'filename': l}
            for l, n in zip(lows, normals)]

test_pairs  = load_image_pairs(LOLV2_PATH / 'Real_captured' / 'Test')
test_loader = DataLoader(LOLDataset(test_pairs), batch_size=1, shuffle=False, num_workers=0)
print(f'Test set: {len(test_pairs)} images')


### 7. Run Evaluation

In [ ]:
metrics = {'psnr': [], 'ssim': [], 'lpips': [], 'niqe': []}

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        low_t  = batch['low'].to(device)
        ref_t  = batch['normal'].to(device)

        alpha    = model(low_t)
        enh_t    = enhance_image_with_curves(low_t, alpha)

        enh_np = enh_t.squeeze(0).cpu().permute(1,2,0).numpy()  # float32 [0,1]
        ref_np = ref_t.squeeze(0).cpu().permute(1,2,0).numpy()

        metrics['psnr'].append(psnr_metric(ref_np, enh_np, data_range=1.0))
        metrics['ssim'].append(ssim_metric(ref_np, enh_np, channel_axis=2, data_range=1.0))

        if LPIPS_OK:
            e2 = enh_t * 2 - 1
            r2 = ref_t * 2 - 1
            metrics['lpips'].append(lpips_model(e2, r2).item())

        if NIQE_OK:
            metrics['niqe'].append(niqe(enh_t.cpu(), data_range=1.0).item())

# Summarise
results = {
    'psnr_mean': float(np.mean(metrics['psnr'])),
    'psnr_std':  float(np.std(metrics['psnr'])),
    'ssim_mean': float(np.mean(metrics['ssim'])),
    'ssim_std':  float(np.std(metrics['ssim'])),
}
if metrics['lpips']:
    results['lpips_mean'] = float(np.mean(metrics['lpips']))
    results['lpips_std']  = float(np.std(metrics['lpips']))
if metrics['niqe']:
    results['niqe_mean'] = float(np.mean(metrics['niqe']))
    results['niqe_std']  = float(np.std(metrics['niqe']))

print('\n' + '='*55)
print('EVALUATION RESULTS')
print('='*55)
for k, v in results.items():
    print(f'  {k:<20}: {v:.4f}')
print('='*55)


### 8. Log Everything to MLflow

This cell logs: hyperparameters, per-epoch losses (as step metrics), final evaluation metrics, training plot, and model checkpoint.


In [ ]:
# Load training history
with open(HISTORY_PATH) as f:
    history = json.load(f)

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f'Run ID: {run.info.run_id}')

    # ── 1. Hyperparameters ─────────────────────────────────────────────
    HPARAMS['n_params'] = sum(p.numel() for p in model.parameters())
    HPARAMS['n_train_images'] = len(test_pairs)  # log test set size too
    mlflow.log_params(HPARAMS)
    print('Logged hyperparameters')

    # ── 2. Per-epoch training losses (step metrics) ────────────────────
    print('Logging per-epoch losses...')
    for epoch, (loss, spa, exp_, col, tv, lr) in enumerate(
        zip(history['loss'], history['spa'], history['exp'],
            history['col'], history['tv'],  history['lr'])
    ):
        mlflow.log_metrics({
            'train/loss_total': loss,
            'train/loss_spa':   spa,
            'train/loss_exp':   exp_,
            'train/loss_col':   col,
            'train/loss_tv':    tv,
            'train/lr':         lr,
        }, step=epoch)
    print(f'Logged {len(history["loss"])} epochs of training metrics')

    # ── 3. Summary training metrics ────────────────────────────────────
    mlflow.log_metrics({
        'train/final_loss': history['loss'][-1],
        'train/best_loss':  min(history['loss']),
        'train/best_epoch': int(np.argmin(history['loss'])) + 1,
    })

    # ── 4. Evaluation metrics ──────────────────────────────────────────
    eval_log = {f'eval/{k}': v for k, v in results.items()}
    mlflow.log_metrics(eval_log)
    print('Logged evaluation metrics')

    # ── 5. Pass/Fail tags ──────────────────────────────────────────────
    mlflow.set_tags({
        'psnr_target_pass':  str(results['psnr_mean'] > 19.0),
        'ssim_target_pass':  str(results['ssim_mean'] > 0.65),
        'lpips_target_pass': str(results.get('lpips_mean', 1) < 0.30),
        'test_dataset':      'LOL-v2 Real_captured Test',
        'framework':         'PyTorch',
    })

    # ── 6. Artifacts ───────────────────────────────────────────────────
    if os.path.exists(TRAINING_PLOT):
        mlflow.log_artifact(TRAINING_PLOT, artifact_path='plots')
        print(f'Logged artifact: {TRAINING_PLOT}')

    mlflow.log_artifact(CHECKPOINT_PATH, artifact_path='model_checkpoint')
    print(f'Logged artifact: {CHECKPOINT_PATH}')

    # ── 7. Log PyTorch model (MLflow model registry format) ────────────
    mlflow.pytorch.log_model(
        model,
        artifact_path='pytorch_model',
        registered_model_name='ZeroDCE-SEBlock',
    )
    print('Logged PyTorch model to registry as ZeroDCE-SEBlock')

    print(f'\nDone! View run at: {MLFLOW_URI}/#/experiments')
    print(f'Run ID: {run.info.run_id}')


### 9. Quick Verify — Pull Run Back from MLflow

In [ ]:
# Sanity check: fetch the run we just logged and print key metrics
client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_URI)
run_data = client.get_run(run.info.run_id)

print('Logged metrics in MLflow:')
for k in ['eval/psnr_mean', 'eval/ssim_mean', 'eval/lpips_mean',
          'train/best_loss', 'train/best_epoch']:
    v = run_data.data.metrics.get(k)
    if v is not None:
        print(f'  {k:<30}: {v:.4f}')

print('\nTags:')
for k, v in run_data.data.tags.items():
    if not k.startswith('mlflow.'):
        print(f'  {k:<30}: {v}')
